# ML Pipeline

erster ChatGPT Draft

In [2]:
"""
WTI Raw
   │
   ▼
WTIFeatureTransformer
   │
   ▼
Target Creation
   │
   ▼
Train/Holdout Split
   │
   ▼
Purged Walk Forward CV
   │
   ├───────────── Fold 1
   ├───────────── Fold 2
   ├───────────── Fold 3
   └───────────── Fold n
            │
            ▼
    TweetFeatureTransformer
            │
            ▼
      TweetAggregator
            │
            ▼
         Merge
            │
            ▼
    Feature Selection
            │
            ▼
        Classifier
"""

'\nWTI Raw\n   │\n   ▼\nWTIFeatureTransformer\n   │\n   ▼\nTarget Creation\n   │\n   ▼\nTrain/Holdout Split\n   │\n   ▼\nPurged Walk Forward CV\n   │\n   ├───────────── Fold 1\n   ├───────────── Fold 2\n   ├───────────── Fold 3\n   └───────────── Fold n\n            │\n            ▼\n    TweetFeatureTransformer\n            │\n            ▼\n      TweetAggregator\n            │\n            ▼\n         Merge\n            │\n            ▼\n    Feature Selection\n            │\n            ▼\n        Classifier\n'

## 1. Purged Time Series Split

Da dein Target

y
t
	​

=1(price
t+h
	​

>price
t
	​

)

ist, überlappen sich Labels.

Bei Horizon=60 Minuten:

In [ ]:
from sklearn.model_selection import BaseCrossValidator
import numpy as np


class PurgedTimeSeriesSplit(BaseCrossValidator):

    def __init__(
        self,
        n_splits=5,
        purge_window=60
    ):
        self.n_splits = n_splits
        self.purge_window = purge_window

    def split(self, X, y=None, groups=None):

        n_samples = len(X)

        fold_size = n_samples // (self.n_splits + 1)

        for i in range(self.n_splits):

            test_start = (i + 1) * fold_size
            test_end = test_start + fold_size

            train_end = test_start - self.purge_window

            train_idx = np.arange(0, train_end)

            test_idx = np.arange(
                test_start,
                min(test_end, n_samples)
            )

            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

## 4. Embedding Transformer

Wichtig:

Embeddings dürfen komplett erzeugt werden.

PCA jedoch nicht.

In [ ]:
from sentence_transformers import (
    SentenceTransformer
)


class EmbeddingTransformer:

    def __init__(self):

        self.model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

    def transform(self, tweets):

        emb = self.model.encode(
            tweets["text"].tolist(),
            show_progress_bar=True
        )

        emb = pd.DataFrame(
            emb,
            index=tweets.index
        )

        return emb

## 5. Tweet Aggregation

Mehrere Tweets pro Minute.

In [ ]:
class TweetAggregator:

    def transform(
        self,
        tweets,
        embeddings
    ):

        tweets["minute"] = (
            tweets.index.floor("min")
        )

        embeddings["minute"] = (
            tweets["minute"]
        )

        sentiment_agg = (

            tweets
            .groupby("minute")
            .agg(
                sentiment_mean=(
                    "sentiment",
                    "mean"
                ),
                sentiment_max=(
                    "sentiment",
                    "max"
                ),
                tweet_count=(
                    "sentiment",
                    "count"
                )
            )
        )

        emb_agg = (
            embeddings
            .groupby("minute")
            .mean()
        )

        return sentiment_agg.join(
            emb_agg
        )

## 6. Merge Transformer

In [ ]:
class TweetMergeTransformer:

    def fit(self, X, y=None):
        return self

    def transform(
        self,
        wti,
        tweet_features
    ):

        merged = wti.merge(
            tweet_features,
            left_index=True,
            right_index=True,
            how="left"
        )

        merged = merged.fillna(0)

        return merged

## 7. Fold-spezifische PCA

Hier passiert die eigentliche Leakage Prävention

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


embedding_cols = [
    c for c in X.columns
    if c.startswith("emb_")
]

Pipeline

In [ ]:
embedding_pipeline = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "pca",
        PCA(
            n_components=20
        )
    )
])

Da PCA innerhalb des sklearn fits liegt

In [ ]:
fit(train)

transform(test)

kein Leakage

## 8. Feature Selection

Innerhalb jedes Folds:

In [ ]:
from sklearn.feature_selection import (
    SelectFromModel
)

Für LogReg

In [ ]:
SelectFromModel(
    LogisticRegression(
        penalty="l1",
        solver="liblinear"
    )
)

## 9. Modellkanidaten

In [ ]:
models = {

    "logreg": LogisticRegression(),

    "xgb": XGBClassifier(),

    "adaboost": AdaBoostClassifier(),

    "mlp": MLPClassifier(
        early_stopping=True
    )
}

## 10. Nested CV + Optuna

In [ ]:
def objective(
    trial,
    X,
    y
):

    params = {

        "max_depth":
        trial.suggest_int(
            "max_depth",
            3,
            8
        ),

        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            1e-3,
            0.2,
            log=True
        )
    }

    splitter = PurgedTimeSeriesSplit(
        n_splits=5,
        purge_window=60
    )

    aucs = []

    for train_idx,test_idx in splitter.split(X):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = XGBClassifier(
            **params,
            eval_metric="auc"
        )

        model.fit(
            X_train,
            y_train
        )

        prob = (
            model
            .predict_proba(X_test)
            [:,1]
        )

        aucs.append(
            roc_auc_score(
                y_test,
                prob
            )
        )

    return np.mean(aucs)

## 11. Baseline vs Tweet Experiment

Automatisch zwei Datensätze erzugen

In [ ]:
baseline_cols = [
    c for c in df.columns
    if not c.startswith("tweet_")
]

X_baseline = df[
    baseline_cols
]

X_tweet = df.copy()

Dann für jedes Modell

In [ ]:
Baseline AUC

vs

Tweet AUC

speichern

## 12. Statistische Signfikanz

Für jede Fold:

auc_baseline = [...]
auc_tweet = [...]

Dann:

In [ ]:
from scipy.stats import wilcoxon

wilcoxon(
    auc_tweet,
    auc_baseline
)

Die eigentliche Forschungsfrage lautet nämlich nicht:

"Kann XGBoost den Ölpreis prognostizieren?"

sondern:

"Verbessern Trump-Tweets die Prognose signifikant gegenüber einem Marktmodell ohne Tweets?"

Deshalb ist der Baseline-vs-Extended-Vergleich mit Purged Walk-Forward CV und anschließendem Signifikanztest methodisch der wichtigste Teil der gesamten Pipeline.